In [ ]:
# ============================================================
# Physics-Informed LSTM — Training Notebook
# Paper: "A Physics-Informed LSTM-Based System Identification
#         Technique for Instantaneous Modal Parameters of
#         Base-Isolated Structures under Seismic Excitation"
#
# HOW TO RUN:
#   1. Run this notebook from the REPOSITORY ROOT directory.
#   2. Set seismic_input below to any case in data/.
#   3. Pre-extracted CNN features (lstm_features_*V2.npy) are
#      already included in data/. Re-run 01_CNN_feature_extraction
#      only if you want to regenerate them.
#   4. Trained weights will be saved in results/<folder_name>/
# ============================================================

import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from scipy.signal import resample
from IPython.display import clear_output
from models import PhysicsInformedLSTM, PhysicsInformedLSTM_Contributions
from utils import normalize, calculate_nrmse 
import cProfile
import time
from SDOF_aceleracion_promedio_tf import SDOF_aceleracion_promedio_tf
from process_window import process_window

# Optional: set threading for multi-core machines (adjust N to your CPU count)
# tf.config.threading.set_intra_op_parallelism_threads(N)
# tf.config.threading.set_inter_op_parallelism_threads(N)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress TF logs
# os.environ['MKL_DEBUG_CPU_TYPE'] = '5'  # AMD Ryzen only


# Extract seismic input name
seismic_input = "BI-BNCS_ICA100"
data_path = os.path.join(os.getcwd(), "data", seismic_input)


# User Settings
MODE = 'train'
LOAD_BEST_WEIGHTS = False # Set to False to avoid shape mismatch; train from scratch with new hidden_dim
PDE_SIGN = -1
WINDOW_SIZE = 100  
NUM_EPOCHS = 7000
LR = 1e-4
ACC_LOSS_WEIGHT = 5
DISPL_LOSS_WEIGHT = 1
PHI_REG_LOSS_WEIGHT = 0  # Regularization to keep phi near baseline PHI
MODE_SUM_LOSS_WEIGHT = 10  # New weight for mode sum constraint
BETA_NM = 0.25
GAMMA_NM = 0.50
Fs = 100
ORIGINAL_FS = 200
dt = 1.0 / Fs
HR = 0.204
USE_TRUE_INPUTS = False
train_mode1_only = False
N_DOF = 6
# 'ACTIVE_MODES' controls how many modes are trained by the LSTM.
# 1: Train only mode 1. Modes 2 and 3 are fixed.
# 2: Train modes 1 and 2. Mode 3 is fixed.
# 3: Train all three modes.
N_MODES = 2
ACTIVE_MODES = N_MODES

# Initial (fixed) parameters for the third mode
initial_damp_mode3 = tf.constant(0.02, dtype=tf.float32) # Example value
initial_freq_mode3 = tf.constant(5.0, dtype=tf.float32)  # Example value

SMOOTH_WEIGHT_DAMP = 1
SMOOTH_WEIGHT_FREQ = 1
DAMP_VARIATION_LIMIT = 0.1
FREQ_VARIATION_LIMIT = 0.1
PLOT_EVERY = 100  # Set to a high number or 0 to disable
VERBOSE = False  # Set to True only for debugging short runs


# Compute weight ratio
total_weight = ACC_LOSS_WEIGHT + DISPL_LOSS_WEIGHT
acc_ratio = round(100*ACC_LOSS_WEIGHT)
displ_ratio = round(100*DISPL_LOSS_WEIGHT)


weight_str = f"acc{acc_ratio}_disp{displ_ratio}"  # Replace \ with _

# Determine initial conditions string
init_str = "AV"  # If needed to set other initial contidion, change manually

# Hidden dim 
current_hidden_dim = 64


# Build folder name
folder_name = f"{seismic_input}_{weight_str}_NM{N_MODES}_NDOF{N_DOF}_{init_str}_Hiddim{current_hidden_dim}"

# Create save directory
save_dir = os.path.join(os.getcwd(), "results", folder_name)
try:
    os.makedirs(save_dir, exist_ok=True)
except OSError as e:
    print(f"Error creating directory {save_dir}: {e}")
    raise

# Precompute mode shapes and modal masses
PHI_full = np.array([
    [1.0000, 1.0000, 1.0000],
    [0.9868, 0.7725, 0.1319],
    [0.9706, 0.4449, -0.76728],
    [0.9451, -0.0509, -0.9784],
    [0.9202, -0.5931, -0.2761],
    [0.8804, -0.9093, 0.4327]
]).astype(np.float32)[:N_DOF, :N_MODES]  # Slice rows to N_DOF, columns to N_MODES
M = tf.convert_to_tensor(np.diag([593.3, 1078.1, 1036.3, 703, 674, (1868.5 + 406.6)][:N_DOF]).astype(np.float32))
R = tf.convert_to_tensor(np.ones((N_DOF, 1), dtype=np.float32))
PHI = tf.convert_to_tensor(PHI_full, dtype=tf.float32)
PHI_normalized = tf.zeros_like(PHI, dtype=tf.float32)
modal_masses = tf.zeros(N_MODES, dtype=tf.float32)
for i in range(N_MODES):
    phi_i = PHI[:, i:i+1]
    modal_mass = tf.reshape(tf.matmul(tf.transpose(phi_i), tf.matmul(M, phi_i)), [])
    modal_masses = tf.tensor_scatter_nd_update(modal_masses, [[i]], [modal_mass])
    participation = tf.squeeze(tf.matmul(tf.transpose(phi_i), tf.matmul(M, R)), axis=[0, 1])
    alpha_i = participation / modal_mass
    indices = tf.constant([[j, i] for j in range(N_DOF)])
    updates = phi_i * alpha_i
    PHI_normalized = tf.tensor_scatter_nd_update(PHI_normalized, indices, tf.reshape(updates, [N_DOF]))
    normalized_modal_mass = tf.matmul(tf.transpose(PHI_normalized[:, i:i+1]), tf.matmul(M, PHI_normalized[:, i:i+1]))
    if VERBOSE:
        print(f"Mode {i+1} pre-normalization participation factor: {participation.numpy():.4f} kg")
        print(f"Mode {i+1} modal mass: {modal_mass.numpy():.4f} kg")
        print(f"Mode {i+1} normalization factor alpha: {alpha_i.numpy():.4f}")
        print(f"Mode {i+1} post-normalization (phi^T M r) / (phi^T M phi): {((tf.squeeze(tf.matmul(tf.transpose(PHI_normalized[:, i:i+1]), tf.matmul(M, R)), axis=[0, 1]) / normalized_modal_mass).numpy().item()):.4f}")
        print(f"Mode {i+1} normalized phi: {PHI_normalized[:, i].numpy()}")
PHI = PHI_normalized


# Define weights paths
weights_path_mode1 = os.path.join(save_dir, f"best_model_mode1_{folder_name}.weights.h5")
weights_path_mode2 = os.path.join(save_dir, f"best_model_mode2_{folder_name}.weights.h5")
weights_path_mode3 = os.path.join(save_dir, f"best_model_mode3_{folder_name}.weights.h5")
weights_path_contrib = os.path.join(save_dir, f"best_model_contrib_{folder_name}.weights.h5")
mid_weights_path_mode1 = os.path.join(save_dir, f"mid_model_mode1_{folder_name}.weights.h5")
mid_weights_path_mode2 = os.path.join(save_dir, f"mid_model_mode2_{folder_name}.weights.h5")
mid_weights_path_mode3 = os.path.join(save_dir, f"mid_model_mode3_{folder_name}.weights.h5")
mid_weights_path_contrib = os.path.join(save_dir, f"mid_model_contrib_{folder_name}.weights.h5")


data_files = [f"a{i}.txt" for i in range(1, N_DOF+1)] + ["u.txt", f"lstm_features_{seismic_input.split('_')[-1]}V2.npy"] + [f"d{i}.txt" for i in range(1, N_DOF+1)] + ["du.txt"]
for file in data_files:
    if not os.path.exists(os.path.join(data_path, file)):
        raise FileNotFoundError(f"Required file {file} not found in {data_path}")
# Load data
accel_data = [np.loadtxt(os.path.join(data_path, f"a{i}.txt")) for i in range(1, N_DOF+1)]
displ_data = [np.loadtxt(os.path.join(data_path, f"d{i}.txt")) for i in range(1, N_DOF+1)]
u = np.loadtxt(os.path.join(data_path, "u.txt"))
cnn_features = np.load(os.path.join(data_path, f"lstm_features_{seismic_input.split('_')[-1]}V2.npy"))
du = np.loadtxt(os.path.join(data_path, "du.txt"))
accel_all = np.stack(accel_data[::-1], axis=1)  # Reverse to match [a6, a5, ..., a1]
displ_all = np.stack(displ_data[::-1], axis=1)

original_timesteps = len(u)
new_timesteps = original_timesteps // 2
accel_all = resample(accel_all, new_timesteps, axis=0)
u = resample(u, new_timesteps)
displ_all = resample(displ_all, new_timesteps, axis=0)
du = resample(du, new_timesteps)
start_sec = 10.0
end_sec = 179.0
start_idx = int(start_sec * Fs)
end_idx = min(int(end_sec * Fs), len(u))
accel_all = accel_all[start_idx:end_idx]
u = u[start_idx:end_idx]
displ_all = displ_all[start_idx:end_idx]
du = du[start_idx:end_idx]
start_win = start_idx // WINDOW_SIZE
end_win = end_idx // WINDOW_SIZE
cnn_features = cnn_features[start_win:end_win]

if VERBOSE:
    print("Files loaded successfully!")
    print("Data shapes:")
    print(" accel_all shape:", accel_all.shape)
    print(" u shape:", u.shape)
    print(" displ_all shape:", displ_all.shape)
    print(" du shape:", du.shape)
    print(" cnn_features shape:", cnn_features.shape)

# Convert to tensors
ground_acc_t = tf.convert_to_tensor(u, dtype=tf.float32)[None, :, None]
accel_all_t = tf.convert_to_tensor(accel_all, dtype=tf.float32)[None, :, :]
displ_all_t = tf.convert_to_tensor(displ_all, dtype=tf.float32)[None, :, :]
du_t = tf.convert_to_tensor(du, dtype=tf.float32)[None, :, None]
total_steps = end_idx - start_idx
num_windows = total_steps // WINDOW_SIZE
if VERBOSE:
    print("Number of windows:", num_windows)
    print("Total steps:", total_steps)

# Initialize models with larger hidden_dim for more compute
model_mode1 = PhysicsInformedLSTM(input_dim=8, hidden_dim=current_hidden_dim, mode=1)
model_mode2 = None if N_MODES < 2 else PhysicsInformedLSTM(input_dim=8, hidden_dim=current_hidden_dim, mode=2)
model_mode3 = None if N_MODES < 3 else PhysicsInformedLSTM(input_dim=8, hidden_dim=current_hidden_dim, mode=3)
model_contrib = None if N_DOF == 1 else PhysicsInformedLSTM_Contributions(input_dim=4 + 4*N_MODES, hidden_dim=current_hidden_dim, n_dof=N_MODES * N_DOF)

if LOAD_BEST_WEIGHTS:
    dummy_input = tf.zeros((1, WINDOW_SIZE, 8), dtype=tf.float32)
    dummy_input_contrib = tf.zeros((1, WINDOW_SIZE, 4 + 4*N_MODES), dtype=tf.float32) if N_DOF > 1 else None
    _ = model_mode1(dummy_input)
    if N_MODES >= 2 and model_mode2:
        _ = model_mode2(dummy_input)
    if N_MODES >= 3 and model_mode3:
        _ = model_mode3(dummy_input)
    if N_DOF > 1 and model_contrib:
        _ = model_contrib(dummy_input_contrib)
    if os.path.exists(weights_path_mode1):
        model_mode1.load_weights(weights_path_mode1)
        if VERBOSE:
            print(f"Loaded best model weights for Mode 1 from {weights_path_mode1}")
    if N_MODES >= 2 and os.path.exists(weights_path_mode2):
        model_mode2.load_weights(weights_path_mode2)
        if VERBOSE:
            print(f"Loaded best model weights for Mode 2 from {weights_path_mode2}")
    if N_MODES >= 3 and os.path.exists(weights_path_mode3):
        model_mode3.load_weights(weights_path_mode3)
        if VERBOSE:
            print(f"Loaded best model weights for Mode 3 from {weights_path_mode3}")
    if N_DOF > 1 and os.path.exists(weights_path_contrib):
        model_contrib.load_weights(weights_path_contrib)
        if VERBOSE:
            print(f"Loaded best model weights for Contributions from {weights_path_contrib}")
else:
    if VERBOSE:
        print("Training all models from scratch")

# Normalize data
max_displ = np.max(np.abs(displ_all))
max_acc = np.max(np.abs(accel_all))
max_seismic = np.max(np.abs(ground_acc_t.numpy()))  # Use .numpy() for max
max_cnn = np.max(np.abs(cnn_features)) if cnn_features.size > 0 else 1.0
displ_all_norm = displ_all / max_displ
accel_all_norm = accel_all / max_acc
ground_acc_t_norm = ground_acc_t / max_seismic
cnn_features_norm = cnn_features / max_cnn if cnn_features.size > 0 else np.zeros((end_win - start_win, WINDOW_SIZE), dtype=np.float32)
displ_all_t_norm = tf.convert_to_tensor(displ_all_norm, dtype=tf.float32)[None, :, :]
accel_all_t_norm = tf.convert_to_tensor(accel_all_norm, dtype=tf.float32)[None, :, :]
np.save('max_displ.npy', max_displ)
np.save('max_acc.npy', max_acc)
np.save('max_seismic.npy', max_seismic)
np.save('max_cnn.npy', max_cnn)

# Pre-slice window inputs
window_inputs = []
for w in range(num_windows):
    start = w * WINDOW_SIZE
    end = start + WINDOW_SIZE
    next_start = (w + 1) * WINDOW_SIZE
    next_end = next_start + WINDOW_SIZE
    current_seismic_norm_tf = ground_acc_t_norm[0, start:end, 0]
    current_cnn_norm_tf = tf.convert_to_tensor(cnn_features_norm[w] if w < len(cnn_features_norm) else np.zeros(WINDOW_SIZE), dtype=tf.float32)
    if next_end <= total_steps:
        next_seismic_norm_tf = ground_acc_t_norm[0, next_start:next_end, 0]
        next_cnn_norm_tf = tf.convert_to_tensor(cnn_features_norm[w+1] if w+1 < len(cnn_features_norm) else np.zeros(WINDOW_SIZE), dtype=tf.float32)
        measured_win_n1_tf = tf.transpose(accel_all_t[0, next_start:next_end, :])
        true_displ_n1_tf = tf.transpose(displ_all_t[0, next_start:next_end, :])
        p_n1 = -ground_acc_t[0, next_start:next_end, 0]
    else:
        next_seismic_norm_tf = tf.zeros([WINDOW_SIZE], dtype=tf.float32)
        next_cnn_norm_tf = tf.zeros([WINDOW_SIZE], dtype=tf.float32)
        measured_win_n1_tf = tf.zeros([N_DOF, WINDOW_SIZE], dtype=tf.float32)
        true_displ_n1_tf = tf.zeros([N_DOF, WINDOW_SIZE], dtype=tf.float32)
        p_n1 = -ground_acc_t[0, start:end, 0]
    window_inputs.append((
        current_seismic_norm_tf, current_cnn_norm_tf, next_seismic_norm_tf, next_cnn_norm_tf,
        measured_win_n1_tf, true_displ_n1_tf, p_n1, p_n1  # p_n1_mode1 and mode2 are same
    ))

# Initialize optimizer and training variables
optimizer = tf.keras.optimizers.Adam(learning_rate=LR)
best_loss = float('inf')
epoch_loss_acc_list = []
epoch_loss_damp_list = []
epoch_loss_freq_list = []
epoch_loss_total_list = []
epoch_loss_displ_list = []
epoch_loss_phi_reg_list = []
epoch_loss_mode_sum_list = []

# --- ADD THIS BLOCK TO DEFINE TENSOR CONSTANTS ---
max_acc_tf = tf.constant(float(max_acc), dtype=tf.float32)
max_displ_tf = tf.constant(float(max_displ), dtype=tf.float32)
Fs_tf = tf.constant(float(Fs), dtype=tf.float32)
ACC_LOSS_WEIGHT_tf = tf.constant(float(ACC_LOSS_WEIGHT), dtype=tf.float32)
DISPL_LOSS_WEIGHT_tf = tf.constant(float(DISPL_LOSS_WEIGHT), dtype=tf.float32)
SMOOTH_WEIGHT_DAMP_tf = tf.constant(float(SMOOTH_WEIGHT_DAMP), dtype=tf.float32)
SMOOTH_WEIGHT_FREQ_tf = tf.constant(float(SMOOTH_WEIGHT_FREQ), dtype=tf.float32)
PHI_REG_LOSS_WEIGHT_tf = tf.constant(float(PHI_REG_LOSS_WEIGHT), dtype=tf.float32)
MODE_SUM_LOSS_WEIGHT_tf = tf.constant(float(MODE_SUM_LOSS_WEIGHT), dtype=tf.float32)
DAMP_VARIATION_LIMIT_tf = tf.constant(float(DAMP_VARIATION_LIMIT), dtype=tf.float32)
FREQ_VARIATION_LIMIT_tf = tf.constant(float(FREQ_VARIATION_LIMIT), dtype=tf.float32)
# --- END BLOCK ---


# Training loop
for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    if VERBOSE:
        profiler = cProfile.Profile()
        profiler.enable()
    modal_y_prev_mode1 = tf.constant(0.0, dtype=tf.float32)
    modal_ydot_prev_mode1 = tf.constant(0.0, dtype=tf.float32)
    prev_damp_seq_mode1 = tf.repeat(0.05, WINDOW_SIZE)
    prev_freq_seq_mode1 = tf.repeat(1.2, WINDOW_SIZE)
    modal_y_prev_mode2 = tf.constant(0.0, dtype=tf.float32)
    modal_ydot_prev_mode2 = tf.constant(0.0, dtype=tf.float32)
    prev_damp_seq_mode2 = tf.repeat(0.025, WINDOW_SIZE)
    prev_freq_seq_mode2 = tf.repeat(3.25, WINDOW_SIZE)
    prev_damp_seq_mode2 = tf.repeat(0.025, WINDOW_SIZE)
    prev_freq_seq_mode2 = tf.repeat(3.25, WINDOW_SIZE)
    modal_y_prev_mode3 = tf.constant(0.0, dtype=tf.float32)
    modal_ydot_prev_mode3 = tf.constant(0.0, dtype=tf.float32)
    prev_damp_seq_mode3 = tf.repeat(initial_damp_mode3, WINDOW_SIZE)
    prev_freq_seq_mode3 = tf.repeat(initial_freq_mode3, WINDOW_SIZE)
    hidden_state_mode1 = None
    hidden_state_mode2 = None
    hidden_state_mode3 = None
    hidden_state_contrib = None
    epoch_grad_norms = []
    epoch_loss = 0.0
    epoch_loss_acc = 0.0
    epoch_loss_damp = 0.0
    epoch_loss_freq = 0.0
    epoch_loss_displ = 0.0
    epoch_loss_phi_reg = 0.0
    epoch_loss_mode_sum = 0.0
    if VERBOSE:
        print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    all_pred_acc = []
    all_true_acc = []
    all_pred_displ = []
    all_true_displ = []
    all_pred_phi_mode1 = []
    all_pred_phi_mode2 = []
    all_pred_phi_mode3 = []
    
    for w in range(num_windows):
        w_tf = tf.constant(w, dtype=tf.int32)
        current_seismic_norm, current_cnn_norm, next_seismic_norm, next_cnn_norm, \
        measured_win_n1, true_displ_n1, p_n1_mode1, p_n1_mode2 = window_inputs[w]
        (
            modal_y_prev_mode1, modal_ydot_prev_mode1, prev_damp_seq_mode1, prev_freq_seq_mode1,
            modal_y_prev_mode2, modal_ydot_prev_mode2, prev_damp_seq_mode2, prev_freq_seq_mode2,
            modal_y_prev_mode3, modal_ydot_prev_mode3, prev_damp_seq_mode3, prev_freq_seq_mode3,
            hidden_state_mode1, hidden_state_mode2, hidden_state_mode3, hidden_state_contrib,
            loss_win, loss_acc_weighted, loss_displ_weighted,
            smooth_loss_damp_weighted, smooth_loss_freq_weighted,
            loss_phi_reg_weighted, loss_mode_sum_weighted, grad_norm_tf,
            a_total_n1, displ_total_n1, phi_mode1_t_trans, phi_mode2_t_trans, phi_mode3_t_trans
        ) = process_window(
            w_tf,
            modal_y_prev_mode1, modal_ydot_prev_mode1, prev_damp_seq_mode1, prev_freq_seq_mode1,
            modal_y_prev_mode2, modal_ydot_prev_mode2, prev_damp_seq_mode2, prev_freq_seq_mode2,
            modal_y_prev_mode3, modal_ydot_prev_mode3, prev_damp_seq_mode3, prev_freq_seq_mode3,
            hidden_state_mode1, hidden_state_mode2, hidden_state_mode3, hidden_state_contrib,
            current_seismic_norm, current_cnn_norm, next_seismic_norm, next_cnn_norm,
            measured_win_n1, true_displ_n1, p_n1_mode1, p_n1_mode2, p_n1_mode2,
            model_mode1, model_mode2, model_mode3, model_contrib,
            optimizer, max_acc_tf, max_displ_tf, Fs_tf, ACC_LOSS_WEIGHT_tf, DISPL_LOSS_WEIGHT_tf,
            SMOOTH_WEIGHT_DAMP_tf, SMOOTH_WEIGHT_FREQ_tf, PHI_REG_LOSS_WEIGHT_tf,
            MODE_SUM_LOSS_WEIGHT_tf, DAMP_VARIATION_LIMIT_tf, FREQ_VARIATION_LIMIT_tf,
            PHI, R, ACTIVE_MODES, WINDOW_SIZE, N_DOF
        )

        grad_norm = grad_norm_tf.numpy()
        epoch_grad_norms.append(grad_norm)
        all_pred_acc.append(a_total_n1.numpy())
        all_true_acc.append(measured_win_n1.numpy())
        all_pred_displ.append(displ_total_n1.numpy())
        all_true_displ.append(true_displ_n1.numpy())
        all_pred_phi_mode1.append(phi_mode1_t_trans.numpy())
        all_pred_phi_mode2.append(phi_mode2_t_trans.numpy())
        all_pred_phi_mode3.append(phi_mode3_t_trans.numpy()) # Correctly appends mode 3
        epoch_loss += loss_win.numpy()
        epoch_loss_acc += loss_acc_weighted.numpy()
        epoch_loss_displ += loss_displ_weighted.numpy()
        epoch_loss_damp += smooth_loss_damp_weighted.numpy()
        epoch_loss_freq += smooth_loss_freq_weighted.numpy()
        epoch_loss_phi_reg += loss_phi_reg_weighted.numpy()
        epoch_loss_mode_sum += loss_mode_sum_weighted.numpy()
        if VERBOSE:
            print(f"Window {w+1}/{num_windows}, grad_norm={grad_norm:.3e}, loss={loss_win.numpy():.3e}, "
                    f"loss_acc={loss_acc_weighted.numpy():.3e}, loss_displ={loss_displ_weighted.numpy():.3e}, "
                    f"loss_phi_reg={loss_phi_reg_weighted.numpy():.3e}, loss_mode_sum={loss_mode_sum_weighted.numpy():.3e}, "
                    f"loss_damp={smooth_loss_damp_weighted.numpy():.3e}, loss_freq={smooth_loss_freq_weighted.numpy():.3e}")

    # Append epoch-wise losses
    epoch_loss_total_list.append(epoch_loss)
    epoch_loss_acc_list.append(epoch_loss_acc)
    epoch_loss_damp_list.append(epoch_loss_damp)
    epoch_loss_freq_list.append(epoch_loss_freq)
    epoch_loss_displ_list.append(epoch_loss_displ)
    epoch_loss_phi_reg_list.append(epoch_loss_phi_reg)
    epoch_loss_mode_sum_list.append(epoch_loss_mode_sum)

    if PLOT_EVERY > 0 and (epoch + 1) % PLOT_EVERY == 0:
        if len(all_pred_acc) > 0:
            pred_acc_epoch = np.concatenate(all_pred_acc, axis=1)
            true_acc_epoch = np.concatenate(all_true_acc, axis=1)
            pred_displ_epoch = np.concatenate(all_pred_displ, axis=1)
            true_displ_epoch = np.concatenate(all_true_displ, axis=1)
            pred_phi_mode1_epoch = np.concatenate(all_pred_phi_mode1, axis=1) if all_pred_phi_mode1 else np.zeros((N_DOF, pred_acc_epoch.shape[1]))
            pred_phi_mode2_epoch = np.concatenate(all_pred_phi_mode2, axis=1) if N_MODES >= 2 and all_pred_phi_mode2 else np.zeros((N_DOF, pred_acc_epoch.shape[1]))
            pred_phi_mode3_epoch = np.concatenate(all_pred_phi_mode3, axis=1) if N_MODES >= 3 and all_pred_phi_mode3 else np.zeros((N_DOF, pred_acc_epoch.shape[1]))
            valid_steps_epoch = pred_acc_epoch.shape[1]
            time_axis_epoch = np.arange(valid_steps_epoch) / Fs + start_sec
            nrmse_acc_epoch = np.array([calculate_nrmse(pred_acc_epoch[i, :], true_acc_epoch[i, :]) for i in range(N_DOF)])
            nrmse_displ_epoch = np.array([calculate_nrmse(pred_displ_epoch[i, :], true_displ_epoch[i, :]) for i in range(N_DOF)])
            nrmse_phi_mode1_epoch = np.array([calculate_nrmse(pred_phi_mode1_epoch[i, :], np.full_like(pred_phi_mode1_epoch[i, :], PHI[i, 0].numpy())) for i in range(N_DOF)]) if N_DOF > 1 else np.array([0.0])
            nrmse_phi_mode2_epoch = np.array([calculate_nrmse(pred_phi_mode2_epoch[i, :], np.full_like(pred_phi_mode2_epoch[i, :], PHI[i, 1].numpy())) for i in range(N_DOF)]) if N_MODES >= 2 and N_DOF > 1 else np.array([0.0])
            nrmse_phi_mode3_epoch = np.array([calculate_nrmse(pred_phi_mode3_epoch[i, :], np.full_like(pred_phi_mode3_epoch[i, :], PHI[i, 2].numpy())) for i in range(N_DOF)]) if N_MODES >= 3 and N_DOF > 1 else np.array([0.0])
            if VERBOSE:
                print(f"Epoch {epoch+1} Monitoring - Avg Acc NRMSE: {np.mean(nrmse_acc_epoch):.4f}, "
                    f"Avg Displ NRMSE: {np.mean(nrmse_displ_epoch):.4f}, "
                    f"Avg Phi Mode1 NRMSE: {np.mean(nrmse_phi_mode1_epoch):.4f}, "
                    f"Avg Phi Mode2 NRMSE: {np.mean(nrmse_phi_mode2_epoch):.4f}, "
                    f"Avg Phi Mode3 NRMSE: {np.mean(nrmse_phi_mode3_epoch):.4f}")
            # Plot Acceleration Comparison
            fig_acc, axs_acc = plt.subplots(N_DOF, 1, figsize=(10, 3 * N_DOF), sharex=True, squeeze=False)
            fig_acc.suptitle(f"Epoch {epoch+1}: Acceleration Comparison")
            for i in range(N_DOF):
                axs_acc[i, 0].plot(time_axis_epoch, true_acc_epoch[i, :], label=f"True CH {i+1}")
                axs_acc[i, 0].plot(time_axis_epoch, pred_acc_epoch[i, :], '--', label=f"Pred CH {i+1}")
                axs_acc[i, 0].set_ylabel(f"CH {i+1} (m/s²)")
                axs_acc[i, 0].legend()
                axs_acc[i, 0].grid(True)
                nrmse_text = f"NRMSE: {nrmse_acc_epoch[i]:.4f}"
                axs_acc[i, 0].text(0.05, 0.85, nrmse_text, transform=axs_acc[i, 0].transAxes)
            axs_acc[-1, 0].set_xlabel("Time (s)")
            plt.tight_layout()
            plt.show()
            # Plot Displacement Comparison
            fig_displ, axs_displ = plt.subplots(N_DOF, 1, figsize=(10, 3 * N_DOF), sharex=True, squeeze=False)
            fig_displ.suptitle(f"Epoch {epoch+1}: Displacement Comparison")
            for i in range(N_DOF):
                axs_displ[i, 0].plot(time_axis_epoch, true_displ_epoch[i, :], label=f"True CH {i+1}")
                axs_displ[i, 0].plot(time_axis_epoch, pred_displ_epoch[i, :], '--', label=f"Pred CH {i+1}")
                axs_displ[i, 0].set_ylabel(f"CH {i+1} (m)")
                axs_displ[i, 0].legend()
                axs_displ[i, 0].grid(True)
                nrmse_text = f"NRMSE: {nrmse_displ_epoch[i]:.4f}"
                axs_displ[i, 0].text(0.05, 0.85, nrmse_text, transform=axs_displ[i, 0].transAxes)
            axs_displ[-1, 0].set_xlabel("Time (s)")
            plt.tight_layout()
            plt.show()
            # Plot Mode Shape Comparison (only for N_DOF > 1)
            if N_DOF > 1:
                fig_phi, axs_phi = plt.subplots(N_DOF, N_MODES, figsize=(12, 3 * N_DOF), sharex=True, squeeze=False)
                fig_phi.suptitle(f"Epoch {epoch+1}: Mode Shape Comparison")
                for i in range(N_DOF):
                    for mode_idx in range(N_MODES):
                        axs_phi[i, mode_idx].plot(time_axis_epoch, np.full_like(time_axis_epoch, PHI[i, mode_idx].numpy()), label=f"Baseline Phi Mode {mode_idx+1} CH {i+1}", color='blue')
                        if mode_idx == 0:
                            axs_phi[i, 0].plot(time_axis_epoch, pred_phi_mode1_epoch[i, :], '--', label=f"Pred Phi Mode 1 CH {i+1}", color='red')
                        elif mode_idx == 1 and N_MODES >= 2:
                            axs_phi[i, 1].plot(time_axis_epoch, pred_phi_mode2_epoch[i, :], '--', label=f"Pred Phi Mode 2 CH {i+1}", color='red')
                        elif mode_idx == 2 and N_MODES >= 3:
                            axs_phi[i, 2].plot(time_axis_epoch, pred_phi_mode3_epoch[i, :], '--', label=f"Pred Phi Mode 3 CH {i+1}", color='red')
                        axs_phi[i, mode_idx].set_ylabel(f"Phi Mode {mode_idx+1} CH {i+1}")
                        axs_phi[i, mode_idx].legend()
                        axs_phi[i, mode_idx].grid(True)
                        nrmse_text = f"NRMSE: {locals()[f'nrmse_phi_mode{mode_idx+1}_epoch'][i]:.4f}"
                        axs_phi[i, mode_idx].text(0.05, 0.85, nrmse_text, transform=axs_phi[i, mode_idx].transAxes)
                    axs_phi[i, -1].set_xlabel("Time (s)")
                plt.tight_layout()
                plt.show()

        if VERBOSE:
            print(f"Epoch {epoch+1}/{NUM_EPOCHS}, Total Loss: {epoch_loss:.4e}, "
                    f"Acc Loss: {epoch_loss_acc:.4e}, Displ Loss: {epoch_loss_displ:.4e}, "
                    f"Phi Reg Loss: {epoch_loss_phi_reg:.4e}, Mode Sum Loss: {epoch_loss_mode_sum:.4e}, "
                    f"Damp Loss: {epoch_loss_damp:.4e}, Freq Loss: {epoch_loss_freq:.4e}")

        # Plotting loss components
        if len(epoch_loss_total_list) > 0 and VERBOSE:
            x_plot = np.arange(1, len(epoch_loss_total_list) + 1)
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 12))
            bottom = np.zeros_like(x_plot, dtype=float)
            ax1.bar(x_plot, epoch_loss_acc_list, bottom=bottom, label="Acc Loss", color='blue')
            bottom += np.array(epoch_loss_acc_list)
            ax1.bar(x_plot, epoch_loss_displ_list, bottom=bottom, label="Displ Loss", color='brown')
            bottom += np.array(epoch_loss_displ_list)
            ax1.bar(x_plot, epoch_loss_phi_reg_list, bottom=bottom, label="Phi Reg Loss", color='purple')
            bottom += np.array(epoch_loss_phi_reg_list)
            ax1.bar(x_plot, epoch_loss_mode_sum_list, bottom=bottom, label="Mode Sum Loss", color='cyan')
            bottom += np.array(epoch_loss_mode_sum_list)
            ax1.bar(x_plot, epoch_loss_damp_list, bottom=bottom, label="Damp Smooth Loss", color='green')
            bottom += np.array(epoch_loss_damp_list)
            ax1.bar(x_plot, epoch_loss_freq_list, bottom=bottom, label="Freq Smooth Loss", color='orange')
            ax1.set_title(f"Epoch vs. Stacked Loss Components - Epoch {epoch+1}")
            ax1.set_xlabel("Epoch")
            ax1.set_ylabel("Loss")
            ax1.legend()
            ax1.grid(True, which="both", ls="--")
            window_indices = np.arange(1, len(epoch_grad_norms) + 1)
            ax2.plot(window_indices, epoch_grad_norms, marker='o', label=f"Epoch {epoch+1} Gradient Norm")
            ax2.set_title(f"Gradient Norm per Window - Epoch {epoch+1}")
            ax2.set_xlabel("Window Index")
            ax2.set_ylabel("Global Gradient Norm")
            ax2.grid(True)
            ax2.legend()
            plt.tight_layout()
            plt.show()

            fig_losses = plt.figure(figsize=(10, 5))
            plt.plot(x_plot, epoch_loss_acc_list, label="Acc Loss")
            plt.plot(x_plot, epoch_loss_displ_list, label="Displ Loss")
            plt.plot(x_plot, epoch_loss_phi_reg_list, label="Phi Reg Loss")
            plt.plot(x_plot, epoch_loss_mode_sum_list, label="Mode Sum Loss")
            plt.plot(x_plot, epoch_loss_damp_list, label="Damp Loss")
            plt.plot(x_plot, epoch_loss_freq_list, label="Freq Loss")
            plt.title(f"Individual Loss Components Over Epochs - Epoch {epoch+1}")
            plt.xlabel("Epoch")
            plt.ylabel("Loss")
            plt.legend()
            plt.grid(True)
            plt.show()

        if (epoch + 1) == NUM_EPOCHS // 2:
            try:
                model_mode1.save_weights(mid_weights_path_mode1)
                if N_MODES >= 2:
                    model_mode2.save_weights(mid_weights_path_mode2)
                if N_MODES >= 3:
                    model_mode3.save_weights(mid_weights_path_mode3)
                if N_DOF > 1:
                    model_contrib.save_weights(mid_weights_path_contrib)
                print(f"Saved mid-training weights at epoch {epoch+1} to {save_dir}")
            except OSError as e:
                print(f"Error saving mid-training weights to {save_dir}: {e}")
                raise
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            try:
                model_mode1.save_weights(weights_path_mode1)
                if N_MODES >= 2:
                    model_mode2.save_weights(weights_path_mode2)
                if N_MODES >= 3:
                    model_mode3.save_weights(weights_path_mode3)
                if N_DOF > 1:
                    model_contrib.save_weights(weights_path_contrib)
                if VERBOSE:
                    print(f"New best loss: {best_loss:.4e} at epoch {epoch+1}, saving model weights to {save_dir}")
            except OSError as e:
                print(f"Error saving best weights to {save_dir}: {e}")
                raise

    print(f"Epoch {epoch+1} time: {time.time() - epoch_start:.2f}s")
    if VERBOSE:
        profiler.disable()
        profiler.print_stats(sort='cumtime')

print("Training complete!")
try:
    model_mode1.save_weights(os.path.join(save_dir, f"physics_informed_lstm_mode1_{folder_name}.weights.h5"))
    if N_MODES >= 2:
        model_mode2.save_weights(os.path.join(save_dir, f"physics_informed_lstm_mode2_{folder_name}.weights.h5"))
    if N_MODES >= 3:
        model_mode3.save_weights(os.path.join(save_dir, f"physics_informed_lstm_mode3_{folder_name}.weights.h5"))
    if N_DOF > 1:
        model_contrib.save_weights(os.path.join(save_dir, f"physics_informed_lstm_contrib_{folder_name}.weights.h5"))
    print(f"Model weights saved to {save_dir}")
    np.savez(os.path.join(save_dir, f"losses_{folder_name}.npz"),
             total=epoch_loss_total_list,
             acc=epoch_loss_acc_list,
             damp=epoch_loss_damp_list,
             freq=epoch_loss_freq_list,
             displ=epoch_loss_displ_list,
             phi_reg=epoch_loss_phi_reg_list,
             mode_sum=epoch_loss_mode_sum_list)
    print(f"Losses per epoch saved to {save_dir}/losses_{folder_name}.npz")
except OSError as e:
    print(f"Error saving final weights or losses to {save_dir}: {e}")
    raise

np.savez(os.path.join(save_dir, f"losses_{folder_name}.npz"),
         total=epoch_loss_total_list,
         acc=epoch_loss_acc_list,
         damp=epoch_loss_damp_list,
         freq=epoch_loss_freq_list,
         displ=epoch_loss_displ_list,
         phi_reg=epoch_loss_phi_reg_list,
         mode_sum=epoch_loss_mode_sum_list)
print(f"Losses per epoch saved to {save_dir}/losses_{folder_name}.npz")

loss_file = os.path.join(save_dir, f"losses_{folder_name}.npz")
if os.path.exists(loss_file):
    loaded_losses = np.load(loss_file)
    epoch_loss_total_list = loaded_losses['total']
    epoch_loss_acc_list = loaded_losses['acc']
    epoch_loss_damp_list = loaded_losses['damp']
    epoch_loss_freq_list = loaded_losses['freq']
    epoch_loss_displ_list = loaded_losses['displ']
    epoch_loss_phi_reg_list = loaded_losses['phi_reg']
    epoch_loss_mode_sum_list = loaded_losses['mode_sum']
    
    # Plotting loss components
    x_plot = np.arange(1, len(epoch_loss_total_list) + 1)
    
    # Stacked bar plot
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 12))
    bottom = np.zeros_like(x_plot, dtype=float)
    ax1.bar(x_plot, epoch_loss_acc_list, bottom=bottom, label="Acc Loss", color='blue')
    bottom += np.array(epoch_loss_acc_list)
    ax1.bar(x_plot, epoch_loss_displ_list, bottom=bottom, label="Displ Loss", color='brown')
    bottom += np.array(epoch_loss_displ_list)
    ax1.bar(x_plot, epoch_loss_phi_reg_list, bottom=bottom, label="Phi Reg Loss", color='purple')
    bottom += np.array(epoch_loss_phi_reg_list)
    ax1.bar(x_plot, epoch_loss_mode_sum_list, bottom=bottom, label="Mode Sum Loss", color='cyan')
    bottom += np.array(epoch_loss_mode_sum_list)
    ax1.bar(x_plot, epoch_loss_damp_list, bottom=bottom, label="Damp Smooth Loss", color='green')
    bottom += np.array(epoch_loss_damp_list)
    ax1.bar(x_plot, epoch_loss_freq_list, bottom=bottom, label="Freq Smooth Loss", color='orange')
    ax1.set_title(f"Epoch vs. Stacked Loss Components - {folder_name}")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.grid(True, which="both", ls="--")
    
    # Individual loss components plot
    ax2.plot(x_plot, epoch_loss_acc_list, label="Acc Loss", color='blue')
    ax2.plot(x_plot, epoch_loss_displ_list, label="Displ Loss", color='brown')
    ax2.plot(x_plot, epoch_loss_phi_reg_list, label="Phi Reg Loss", color='purple')
    ax2.plot(x_plot, epoch_loss_mode_sum_list, label="Mode Sum Loss", color='cyan')
    ax2.plot(x_plot, epoch_loss_damp_list, label="Damp Loss", color='green')
    ax2.plot(x_plot, epoch_loss_freq_list, label="Freq Loss", color='orange')
    ax2.set_title(f"Individual Loss Components Over Epochs - {folder_name}")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Loss")
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"loss_plots_{folder_name}.png"))
    plt.show()
    print(f"Loss plots saved to {save_dir}/loss_plots_{folder_name}.png")
else:
    print(f"Loss file {loss_file} not found, skipping loss plotting.")